# AlphaBuilder v4.1 — Etapa 6: Self-Play Curricular (`Phys-UViT 3D`, Kaggle `2 x Tesla T4`)

Notebook oficial da **Etapa 6 (Plano de Ataque v4.1 ao SOTA Discreto vs. BESO 3D)** pronto para executar no Kaggle (`2 x Tesla T4`, ~10.5h úteis dentro do limite de 12h).

### Datasets / Fontes obrigatórias para anexar no Kaggle
1. Dataset de aquecimento: `gustavomello9600/alphabuilder-v3-3-warmup-4k` (`warmup_v3_3_kaggle.db` + `split_v3_3.json`)
2. Saída Stage 5 (para `START_GEN = 1`): `gustavomello9600/alphabuilder-v4-stage5-phys-uvit-part2` (ou `alphabuilder-v4-stage5-phys-uvit`, contendo `best_model.pt` do `Phys-UViT 3D`)
3. *(Opcional Part 2)* Ao retomar de uma execução anterior do Stage 6 (`START_GEN > 1`), anexe o kernel anterior contendo `phys_uvit_selfplay_gen*.pt`, `selfplay_gen*.db` e `selfplay_evolution_history.json`.

### Arquitetura executada
- **Gate 0 (Ablação Zero-Retrain `Gen 0 + v4.1`)**: mede o ganho puro de algoritmo antes de atualizar qualquer peso.
- **Motor `MCTS-DAG v4.1`**: FPU relativo ao pai, Fase 1 unitaria multi-camada com `DAG` e PBRS geodésico c/ decaimento, e Fase 2 com remoção restrita à superfície + veto BFS em lote, macro-ações `ADD/REM/SWAP/STOP`, tabela de transposições e reuso de subárvore.
- **Ciclo de Vida Blindado**: *Grace Period* (5 passos), lotes adaptativos (`16 -> 8 -> 4 -> 2`), parada por paciência em `R_total` + detecção de ciclos + retropropagação protegida pelo pico (`best-of-episode`).
- **Cabeça `op_head` compatível com Stage 5 (`strict=False`)**: treinada com `--lambda-op 0.20` via DDP sobre replay misto `60% Self-Play / 40% Warmup`, avaliada em `S1-S6` e `G1-G4` OOD.

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

SESSION_T0 = time.time()
# Evita contencao de threads entre 2 workers PyAMG nos 4 vCPUs do Kaggle
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("AB_FEM_BACKEND", "fast_amg")

import torch

print("=" * 72)
print("AlphaBuilder v4.1 - Stage 6 Self-Play Campaign (Kaggle 2 x Tesla T4)")
print("=" * 72)
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()} | GPU count: {torch.cuda.device_count()}")
for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f"  GPU {idx}: {torch.cuda.get_device_name(idx)} ({props.total_memory / 1024**3:.1f} GB)")


In [ ]:
# === Configuracao da Campanha Stage 6 v4.1 ===
MODE = "full"       # "full" = campanha real | "smoke" = validacao rapida (~30-38 min + Gate 0 completo)
START_GEN = 1               # Para Part 2 no Kaggle, mude para (ultima_geracao_concluida + 1)
END_GEN = 1 if MODE == "smoke" else 5
RUN_GATE0_ABLATION = (START_GEN == 1)  # Executa Gate 0 (Gen 0 + v4.1 sem retreino) antes da Gen 1
EPISODES_PER_GEN = 8 if MODE == "smoke" else 64
TRAIN_EPOCHS = 2 if MODE == "smoke" else 6
BATCH_SIZE = 32             # Por GPU -> global_bs=64 em 2x T4 (~4.9 GB VRAM)
NUM_SIMULATIONS = 64 if MODE == "smoke" else 160
MAX_PHASE1_STEPS = 80 if MODE == "smoke" else 100
MAX_PHASE2_STEPS = 30 if MODE == "smoke" else 64
LEARNING_RATE = 5e-5
SELFPLAY_FRACTION = 0.60
LAMBDA_OP = 0.20
SESSION_MAX_WALL_HOURS = 0.80 if MODE == "smoke" else 10.5

REPO_URL = "https://github.com/gustavomello9600/alphabuild.git"
REPO_DIR = Path("/tmp/alphabuild")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

try:
    import pyamg  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyamg"], check=True)
    import pyamg  # noqa: F401

# Smoke import check sem FEniCSx (valida fast_amg + MCTS-DAG v4.1)
from alphabuilder.src.logic.selfplay.evaluator import HAS_FAST_FEM, resolve_fem_backend
print("Active commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print(f"PyAMG={pyamg.__version__} | HAS_FAST_FEM={HAS_FAST_FEM} | backend={resolve_fem_backend('fast_amg')}")


In [ ]:
# === Descoberta Inteligente de Warmup DB, Split e Checkpoint Stage 5/6 ===
import json
import re
import shutil

search_roots = [Path("/kaggle/input"), REPO_DIR / "checkpoints", REPO_DIR / "data" / "compiled", Path("/kaggle/working")]
WORK_DIR = Path("/kaggle/working/stage6_work")
(WORK_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "eval").mkdir(parents=True, exist_ok=True)

def _find_all(patterns):
    out = []
    for pat in patterns:
        for root in search_roots:
            if root.exists():
                out.extend(sorted(root.rglob(pat)))
    # Dedup preservando ordem
    seen = set()
    uniq = []
    for p in out:
        rp = str(p.resolve())
        if rp not in seen:
            seen.add(rp)
            uniq.append(p)
    return uniq

dbs = _find_all(["warmup_v3_3_kaggle.db"])
splits = _find_all(["split_v3_3.json"])
if not dbs:
    raise FileNotFoundError("warmup_v3_3_kaggle.db nao encontrado! Anexe o dataset gustavomello9600/alphabuilder-v3-3-warmup-4k.")
src_db = dbs[0]
split_path = splits[0] if splits else None

LOCAL_DB_PATH = Path("/tmp/warmup_v3_3_kaggle.db")
if not LOCAL_DB_PATH.exists() or LOCAL_DB_PATH.stat().st_size != src_db.stat().st_size:
    t0 = time.perf_counter()
    print(f"Copying {src_db} -> {LOCAL_DB_PATH} ({src_db.stat().st_size / 1e9:.2f} GB)...")
    shutil.copyfile(src_db, LOCAL_DB_PATH)
    print(f"Copied in {time.perf_counter() - t0:.1f}s")

def _pick_best_phys_uvit_ckpt(prefer_gen=None):
    patterns = [f"phys_uvit_selfplay_gen{prefer_gen}.pt", "phys_uvit_selfplay_gen*.pt"] if prefer_gen is not None else []
    patterns.extend(["phys_uvit_v4_stage5_best.pt", "best_model.pt"])
    cands = _find_all(patterns)
    valid = []
    for p in cands:
        try:
            hdr = torch.load(p, map_location="cpu", weights_only=False)
        except Exception as exc:
            print(f"  [skip] {p}: {exc}")
            continue
        cfg = hdr.get("config", {}) if isinstance(hdr, dict) else {}
        if str(cfg.get("architecture", "")) != "phys_uvit":
            continue
        ep = int(hdr.get("epoch", -1)) if isinstance(hdr, dict) else -1
        vl = float(hdr.get("val_loss", float("inf"))) if isinstance(hdr, dict) else float("inf")
        is_exact_gen = bool(prefer_gen is not None and p.name == f"phys_uvit_selfplay_gen{prefer_gen}.pt")
        valid.append((1 if is_exact_gen else 0, ep, -vl, str(p), p))
    if not valid:
        raise FileNotFoundError("Nenhum checkpoint PhysUViT3D valido encontrado em /kaggle/input!")
    valid.sort(reverse=True)
    best = valid[0]
    print(f"Selected init checkpoint: {best[4]} (epoch={best[1]}, val_loss={-best[2]:.4f})")
    return best[4]

init_ckpt = _pick_best_phys_uvit_ckpt(prefer_gen=(START_GEN - 1) if START_GEN > 1 else None)

# Dry-check de compatibilidade Stage 5 -> Stage 6 v4.1 (op_head zero-init via strict=False)
from alphabuilder.src.neural.inference import AlphaBuilderInference
_probe_m = AlphaBuilderInference(init_ckpt, device="cpu")
print(f"Compatibility check OK: arch={_probe_m.architecture}, epoch={_probe_m.epoch}, op_head_params={sum(p.numel() for p in _probe_m.model.op_head.parameters())}")
del _probe_m

# Retomada automatica de bancos e historico caso START_GEN > 1
prior_dbs_raw = _find_all(["selfplay_gen*.db"])
def _gen_num(p):
    m = re.search(r"gen(\d+)", p.stem)
    return int(m.group(1)) if m else 0
prior_dbs = [str(p) for p in sorted(prior_dbs_raw, key=_gen_num) if _gen_num(p) < START_GEN]
if START_GEN > 1:
    for hist_cand in _find_all(["selfplay_evolution_history.json"]):
        hist_data = json.loads(hist_cand.read_text(encoding="utf-8"))
        kept_hist = [h for h in hist_data if int(h.get("generation", 0)) < START_GEN]
        if kept_hist:
            (WORK_DIR / "selfplay_evolution_history.json").write_text(json.dumps(kept_hist, indent=2), encoding="utf-8")
            print(f"Restored prior Stage 6 history ({len(kept_hist)} gens) from {hist_cand}")
            break

print(f"Warmup DB : {LOCAL_DB_PATH}")
print(f"Split JSON: {split_path}")
print(f"Init Ckpt : {init_ckpt}")
print(f"Prior SP  : {prior_dbs}")


In [ ]:
# === Execucao: Gate 0 (Zero-Retrain Gen 0 + v4.1 em paralelo) + Orquestrador Stage 6 ===
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
eval_dev = "cuda" if n_gpus > 0 else "cpu"
gate0_json = WORK_DIR / "eval" / "eval_gen0_v41_canonical.json"

if RUN_GATE0_ABLATION and START_GEN == 1:
    print("=" * 72)
    print("Running Gate 0: Zero-Retrain Ablation (Gen 0 + MCTS-DAG v4.1, S1-S6 @ 160 sims x 30 steps)")
    print("=" * 72)
    g0_shards = [
        ["S1_cantilever_long_center", "S2_cantilever_tip_bottom", "S3_cantilever_midspan_top"],
        ["S4_rail_xy_center", "S5_rail_xy_3d_eccentric", "S6_short_deep_clamp"],
    ] if n_gpus >= 2 else [[
        "S1_cantilever_long_center", "S2_cantilever_tip_bottom", "S3_cantilever_midspan_top",
        "S4_rail_xy_center", "S5_rail_xy_3d_eccentric", "S6_short_deep_clamp",
    ]]
    g0_procs = []
    g0_json_parts = []
    for r, sc_list in enumerate(g0_shards):
        part_json = WORK_DIR / "eval" / f"eval_gen0_v41_shard{r}.json"
        part_db = WORK_DIR / "eval" / f"eval_gen0_v41_shard{r}.db"
        g0_json_parts.append(part_json)
        g0_env = os.environ.copy()
        if n_gpus > 0:
            g0_env["CUDA_VISIBLE_DEVICES"] = str(r)
        g0_cmd = [
            sys.executable,
            "scripts/evaluate_inference_scenarios.py",
            "--checkpoint", str(init_ckpt),
            "--prefix", "gen0_v41",
            "--checkpoint-version", "phys-uvit-v4-stage5-v41",
            "--db-path", str(part_db),
            "--json-out", str(part_json),
            "--device", eval_dev,
            "--num-simulations", "160",
            "--max-phase1-steps", "80",
            "--max-phase2-steps", "30",
            "--suite", "canonical",
            "--scenarios", ",".join(sc_list),
            "--fem-backend", "fast_amg",
            "--v41-mcts",
        ]
        print(f"  [Gate 0 GPU {r}] {', '.join(sc_list)}")
        g0_procs.append(subprocess.Popen(g0_cmd, env=g0_env))
    for r, p in enumerate(g0_procs):
        if p.wait() != 0:
            raise RuntimeError(f"Gate 0 shard {r} exited with code {p.returncode}")
    merged_eps = []
    base_payload = None
    for pj in g0_json_parts:
        pl = json.loads(pj.read_text(encoding="utf-8"))
        if base_payload is None:
            base_payload = pl
        merged_eps.extend(pl.get("episodes", []))
    if base_payload is None:
        base_payload = {}
    base_payload["episodes"] = merged_eps
    gate0_json.write_text(json.dumps(base_payload, indent=2), encoding="utf-8")
    if merged_eps:
        import numpy as np
        for ep in merged_eps:
            md = ep["mcts_diagnostics"]
            c_str = f"{ep['final_compliance']:.1f}" if ep.get("final_compliance") is not None else "---"
            print(f"  [GATE 0] {ep['scenario_id']}: Score={ep['final_score']:+.3f} Best={ep.get('best_score', ep['final_score']):+.3f} C={c_str} V={ep['final_volume']*100:.2f}% ADD={md['p2_add_actions_total']} REM={md['p2_rem_actions_total']}")
        g0_score = float(np.mean([e["final_score"] for e in merged_eps]))
        g0_best = float(np.mean([e.get("best_score", e["final_score"]) for e in merged_eps]))
        g0_add = sum(int(e["mcts_diagnostics"]["p2_add_actions_total"]) for e in merged_eps)
        g0_rem = sum(int(e["mcts_diagnostics"]["p2_rem_actions_total"]) for e in merged_eps)
        g0_rem_pct = 100.0 * g0_rem / max(1, g0_add + g0_rem)
        print(f"[GATE 0 SUMMARY] S1-S6 MeanScore={g0_score:+.3f} (Best={g0_best:+.3f}) | REM%={g0_rem_pct:.1f}% (vs Gen 0 v4.0: Score=+0.076, REM=0.35%)")

elapsed_h = (time.time() - SESSION_T0) / 3600.0
rem_wall_h = max(0.25, round(SESSION_MAX_WALL_HOURS - elapsed_h, 2))
print(f"Remaining wall-clock budget for Self-Play gens: {rem_wall_h:.2f}h")

cmd = [
    sys.executable,
    "scripts/run_selfplay_pipeline.py",
    "--mode", MODE,
    "--init-checkpoint", str(init_ckpt),
    "--warmup-db", str(LOCAL_DB_PATH),
    "--work-dir", str(WORK_DIR),
    "--start-gen", str(START_GEN),
    "--end-gen", str(END_GEN),
    "--episodes-per-gen", str(EPISODES_PER_GEN),
    "--train-epochs", str(TRAIN_EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--lr", str(LEARNING_RATE),
    "--selfplay-fraction", str(SELFPLAY_FRACTION),
    "--lambda-op", str(LAMBDA_OP),
    "--num-simulations", str(NUM_SIMULATIONS),
    "--max-phase1-steps", str(MAX_PHASE1_STEPS),
    "--max-phase2-steps", str(MAX_PHASE2_STEPS),
    "--fem-backend", "fast_amg",
    "--max-wall-hours", str(rem_wall_h),
]
if split_path is not None:
    cmd.extend(["--split-json", str(split_path)])
if prior_dbs:
    cmd.extend(["--prior-selfplay-dbs", ",".join(prior_dbs)])

print("Running Stage 6 Pipeline:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# === Resumo da Evolucao, Graficos vs Baselines BESO 3D/SIMP & Exportacao ===
OUT_ROOT = Path("/kaggle/working")
hist_file = WORK_DIR / "selfplay_evolution_history.json"
gate0_json = WORK_DIR / "eval" / "eval_gen0_v41_canonical.json"

gate0_score = None
if gate0_json.exists():
    g0_eps = json.loads(gate0_json.read_text(encoding="utf-8")).get("episodes", [])
    if g0_eps:
        import numpy as np
        gate0_score = float(np.mean([e["final_score"] for e in g0_eps]))
        print(f"Gate 0 (Gen 0 + v4.1, n={len(g0_eps)}): MeanScore={gate0_score:+.3f}")

if hist_file.exists():
    history = json.loads(hist_file.read_text(encoding="utf-8"))
    for entry in history:
        g = entry["generation"]
        c30 = entry.get("eval_canonical_30x8") or {}
        ext = entry.get("eval_canonical_extended") or {}
        ood = entry.get("eval_ood_g1_g4") or {}
        gs = entry.get("generation_stats") or {}
        print(
            f"Gen {g} ({entry['cumulative_gpu_hours']:.2f} GPU-h): "
            f"S1-S6(30x8) Score={c30.get('mean_score', 0):+.3f} (Best={gs.get('mean_best_score', 0):+.3f}), "
            f"C={c30.get('mean_compliance')}, logC_MAE={c30.get('log_c_ood_mae')}, "
            f"REM%={gs.get('p2_rem_fraction', 0)*100:.1f}%, STOP%={gs.get('stop_fraction', 0)*100:.1f}%, "
            f"Reuse={gs.get('mean_tree_reuse_hits', 0):.1f}, DAG={gs.get('total_dag_transpositions', 0)} | "
            f"ExtScore={ext.get('mean_score', '---')} | OOD={ood.get('mean_score', '---')}"
        )

    try:
        import matplotlib.pyplot as plt
        gens = [h["generation"] for h in history]
        fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8))
        axes[0].axhline(0.6235, color="#0F766E", linestyle="--", lw=1.4, label="BESO 3D Pareto (+0.623)")
        axes[0].axhline(0.6074, color="#1D4ED8", linestyle="-.", lw=1.3, label="BESO 3D iso-V (+0.607)")
        axes[0].axhline(0.5092, color="#D97706", linestyle="--", lw=1.3, label="SIMP_cont (+0.509)")
        axes[0].axhline(0.0762, color="#94A3B8", linestyle=":", lw=1.2, label="Gen 0 v4.0 (+0.076)")
        if gate0_score is not None:
            axes[0].scatter([0], [gate0_score], color="#B91C1C", s=75, zorder=5, label="Gate 0 (+v4.1)")
        axes[0].plot(gens, [h["eval_canonical_30x8"]["mean_score"] for h in history], "o-", lw=2, label="S1-S6 (30x8)")
        if all(h.get("eval_canonical_extended") for h in history):
            axes[0].plot(gens, [h["eval_canonical_extended"]["mean_score"] for h in history], "s-", lw=2, label="S1-S6 (64 steps)")
        if all(h.get("eval_ood_g1_g4") for h in history):
            axes[0].plot(gens, [h["eval_ood_g1_g4"]["mean_score"] for h in history], "^--", lw=1.8, label="G1-G4 OOD")
        axes[0].set_title("Score v3.3 vs Baselines SOTA")
        axes[0].set_xlabel("Geracao Self-Play")
        axes[0].legend(fontsize=7.5); axes[0].grid(True, alpha=0.3)

        axes[1].plot(gens, [h["eval_canonical_30x8"]["log_c_ood_mae"] for h in history], "o-", label="log10(C) OOD MAE")
        axes[1].plot(gens, [h["eval_canonical_30x8"]["value_ood_mae"] for h in history], "s-", label="Value OOD MAE")
        axes[1].set_title("Calibracao OOD")
        axes[1].set_xlabel("Geracao Self-Play")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)

        axes[2].plot(gens, [h["generation_stats"]["p2_rem_fraction"] * 100 for h in history], "o-", label="REMOVE% (Fase 2)")
        axes[2].plot(gens, [h["eval_canonical_30x8"]["mean_hollow_xy"] * 100 for h in history], "s-", label="Hollow% (S1-S6)")
        axes[2].plot(gens, [h["generation_stats"].get("stop_fraction", 0) * 100 for h in history], "^--", label="STOP% (Fase 2)")
        axes[2].set_title("Escavacao, Trelica & Parada Ativa")
        axes[2].set_xlabel("Geracao Self-Play")
        axes[2].legend(); axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(OUT_ROOT / "stage6_evolution.png", dpi=150)
        plt.show()
    except Exception as exc:
        print("Plot skipped:", exc)

    shutil.copy2(hist_file, OUT_ROOT / "selfplay_evolution_history.json")

# Exporta checkpoints, ultimos 5 DBs para retomada Part 2 e JSONs de avaliacao
eval_export = OUT_ROOT / "eval_reports"
eval_export.mkdir(parents=True, exist_ok=True)
for js in sorted((WORK_DIR / "eval").glob("*.json")):
    shutil.copy2(js, eval_export / js.name)

for pt in sorted((WORK_DIR / "checkpoints").glob("*.pt")):
    shutil.copy2(pt, OUT_ROOT / pt.name)
    print(f"Exported ckpt: /kaggle/working/{pt.name} ({pt.stat().st_size / 1e6:.2f} MB)")

sp_dbs_sorted = sorted(WORK_DIR.glob("selfplay_gen*.db"))
for dbp in sp_dbs_sorted[-5:]:
    shutil.copy2(dbp, OUT_ROOT / dbp.name)
    print(f"Exported DB  : /kaggle/working/{dbp.name} ({dbp.stat().st_size / 1e6:.2f} MB)")

print("Stage 6 Kaggle pipeline finished cleanly.")
